In [0]:
# ============================================================
# BRONZE AUTO LOADER INGESTION
# ============================================================
#
# Architecture:
#
# Local Folder
#      ↓
# Watcher Script
#      ↓
# Unity Catalog Volume
#      ↓
# Auto Loader
#      ↓
# foreachBatch
#      ↓
# Filename Identification
#      ↓
# Appropriate Bronze Delta Table
#
# Example:
#
# crm/cust_info.csv
#       ↓
# de_project.bronze.cust_info
#
# crm/cust_info_2026-09-03.csv
#       ↓
# de_project.bronze.cust_info
#
# erp/cust_az12.csv
#       ↓
# de_project.bronze.cust_az12
#
# erp/cust_az12_2026-09-03.csv
#       ↓
# de_project.bronze.cust_az12
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

from pyspark.sql import functions as F
import re


# ============================================================
# 2. CONFIGURATION
# ============================================================

# Root of your Unity Catalog Volume
SOURCE_PATH = "/Volumes/de_project/volume/landing"


# Bronze schema
BRONZE_SCHEMA = "de_project.bronze"


# Auto Loader checkpoint location
CHECKPOINT_PATH = (
    "/Volumes/de_project/volume/landing/"
    "_checkpoints/bronze_autoloader"
)


# ============================================================
# 3. CREATE BRONZE SCHEMA
# ============================================================

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}
""")


print(f"Bronze schema ready: {BRONZE_SCHEMA}")


# ============================================================
# 4. FILE -> BRONZE TABLE ROUTING
# ============================================================
#
# The key represents the expected dataset name.
#
# The value represents the Bronze table.
#
# Example:
#
# cust_info.csv
#       ↓
# cust_info
#
# cust_info_2026-09-03.csv
#       ↓
# cust_info
#
# ============================================================

TABLE_ROUTING = {

    # -------------------------
    # CRM
    # -------------------------

    "cust_info": "cust_info",

    "sales_details": "sales_details",

    "prd_info": "prd_info",


    # -------------------------
    # ERP
    # -------------------------

    "cust_az12": "cust_az12",

    "loc_a101": "loc_a101",

    "px_cat_g1v2": "px_cat_g1v2"
}


# ============================================================
# 5. IDENTIFY BRONZE TABLE FROM FILE NAME
# ============================================================

def get_table_name(file_path):

    """
    Determine the Bronze table from the incoming filename.

    Supports:

        cust_info.csv

    and:

        cust_info_2026-09-03.csv

    and:

        cust_info_2026-09-04.csv

    """

    # --------------------------------------------------------
    # Get filename from full path
    # --------------------------------------------------------

    file_name = file_path.split("/")[-1]


    # --------------------------------------------------------
    # Remove .csv extension
    # --------------------------------------------------------

    file_stem = re.sub(
        r"\.csv$",
        "",
        file_name,
        flags=re.IGNORECASE
    )


    # --------------------------------------------------------
    # Convert to lowercase
    # --------------------------------------------------------

    file_stem = file_stem.lower()


    # --------------------------------------------------------
    # Check against known datasets
    #
    # Longest names are checked first.
    #
    # --------------------------------------------------------

    for table_name in sorted(
        TABLE_ROUTING.keys(),
        key=len,
        reverse=True
    ):

        # ----------------------------------------------------
        # CASE 1
        #
        # Current filename
        #
        # cust_info.csv
        #
        # ----------------------------------------------------

        if file_stem == table_name:

            return TABLE_ROUTING[table_name]


        # ----------------------------------------------------
        # CASE 2
        #
        # Future dated filename
        #
        # cust_info_2026-09-03.csv
        #
        # ----------------------------------------------------

        if file_stem.startswith(table_name + "_"):

            return TABLE_ROUTING[table_name]


    # --------------------------------------------------------
    # No matching dataset
    # --------------------------------------------------------

    return None


# ============================================================
# 6. IDENTIFY SOURCE SYSTEM
# ============================================================

def get_source_system(file_path):

    """
    Determine whether the file came from CRM or ERP.
    """

    path_lower = file_path.lower()


    if "/crm/" in path_lower:

        return "crm"


    if "/erp/" in path_lower:

        return "erp"


    return "unknown"


# ============================================================
# 7. PROCESS EACH MICRO-BATCH
# ============================================================

def process_batch(batch_df, batch_id):

    """
    Process one Auto Loader micro-batch.

    Auto Loader detects files.

    foreachBatch receives those files.

    Each file is processed separately because
    CRM and ERP datasets have different schemas.
    """

    print("=" * 70)

    print(f"Processing batch: {batch_id}")

    print("=" * 70)


    # ========================================================
    # CHECK IF BATCH IS EMPTY
    # ========================================================

    if batch_df.isEmpty():

        print("Batch is empty.")

        return


    # ========================================================
    # GET FILES IN THIS BATCH
    # ========================================================

    files = (
        batch_df
        .select(
            "_source_path",
            "_source_file"
        )
        .distinct()
        .collect()
    )


    print(f"Files detected: {len(files)}")


    # ========================================================
    # PROCESS EACH FILE
    # ========================================================

    for file_row in files:

        source_path = file_row["_source_path"]

        source_file = file_row["_source_file"]


        print()
        print("-" * 70)
        print(f"Processing file: {source_file}")
        print(f"Source path: {source_path}")
        print("-" * 70)


        # ====================================================
        # IDENTIFY SOURCE SYSTEM
        # ====================================================

        source_system = get_source_system(source_path)


        print(
            f"Source system: {source_system}"
        )


        # ====================================================
        # IDENTIFY TARGET BRONZE TABLE
        # ====================================================

        table_name = get_table_name(source_path)


        # ====================================================
        # UNKNOWN FILE
        # ====================================================

        if table_name is None:

            print(
                f"WARNING: No table mapping found for "
                f"file: {source_file}"
            )

            print(
                "File will NOT be loaded into Bronze."
            )

            continue


        # ====================================================
        # BUILD TARGET TABLE NAME
        # ====================================================

        target_table = (
            f"{BRONZE_SCHEMA}.{table_name}"
        )


        print(
            f"Target Bronze table: {target_table}"
        )


        # ====================================================
        # READ THE ACTUAL CSV FILE
        # ====================================================
        #
        # Each CSV can have a different schema.
        #
        # Therefore we read the individual file here.
        #
        # ====================================================

        file_df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .option("mode", "PERMISSIVE")
            .load(source_path)
        )


        # ====================================================
        # CHECK IF FILE CONTAINS DATA
        # ====================================================

        record_count = file_df.count()


        print(
            f"Records detected: {record_count}"
        )


        if record_count == 0:

            print(
                f"WARNING: {source_file} contains "
                f"no records."
            )

            continue


        # ====================================================
        # ADD BRONZE METADATA
        # ====================================================

        file_df = (

            file_df

            # ------------------------------------------------
            # Original filename
            # ------------------------------------------------

            .withColumn(
                "_source_file",
                F.lit(source_file)
            )

            # ------------------------------------------------
            # Full source path
            # ------------------------------------------------

            .withColumn(
                "_source_path",
                F.lit(source_path)
            )

            # ------------------------------------------------
            # CRM / ERP
            # ------------------------------------------------

            .withColumn(
                "_source_system",
                F.lit(source_system)
            )

            # ------------------------------------------------
            # When Databricks ingested the file
            # ------------------------------------------------

            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )
        )


        # ====================================================
        # CREATE BRONZE TABLE IF IT DOES NOT EXIST
        # ====================================================

        if not spark.catalog.tableExists(target_table):

            print(
                f"Bronze table does not exist."
            )

            print(
                f"Creating: {target_table}"
            )


            (
                file_df
                .write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table)
            )


        # ====================================================
        # TABLE ALREADY EXISTS
        # ====================================================

        else:

            print(
                f"Bronze table already exists."
            )

            print(
                f"Appending data to: {target_table}"
            )


            (
                file_df
                .write
                .format("delta")
                .mode("append")
                .option(
                    "mergeSchema",
                    "true"
                )
                .saveAsTable(target_table)
            )


        # ====================================================
        # SUCCESS MESSAGE
        # ====================================================

        print(
            f"SUCCESS: {record_count} records loaded "
            f"from {source_file} "
            f"into {target_table}"
        )


    # ========================================================
    # BATCH COMPLETE
    # ========================================================

    print()
    print("=" * 70)

    print(
        f"Batch {batch_id} completed successfully."
    )

    print("=" * 70)


# ============================================================
# 8. CREATE AUTO LOADER STREAM
# ============================================================
#
# We use binaryFile because the CRM and ERP files can
# have completely different schemas.
#
# Auto Loader's responsibility here is:
#
#     "Tell us when a new file has arrived."
#
# foreachBatch's responsibility is:
#
#     "Determine what the file is and load it correctly."
#
# ============================================================

raw_stream = (

    spark.readStream

    .format("cloudFiles")

    # --------------------------------------------------------
    # Read files as binary
    # --------------------------------------------------------

    .option(
        "cloudFiles.format",
        "binaryFile"
    )

    # --------------------------------------------------------
    # Process existing files on the first run
    # --------------------------------------------------------

    .option(
        "cloudFiles.includeExistingFiles",
        "true"
    )

    # --------------------------------------------------------
    # Auto Loader schema metadata
    # --------------------------------------------------------

    .option(
        "cloudFiles.schemaLocation",
        f"{CHECKPOINT_PATH}/schema"
    )

    # --------------------------------------------------------
    # Watch entire landing Volume
    #
    # CRM + ERP
    #
    # --------------------------------------------------------

    .load(SOURCE_PATH)
)


# ============================================================
# 9. SELECT FILE METADATA
# ============================================================

stream_df = (

    raw_stream

    .select(

        # ----------------------------------------------------
        # Full path
        # ----------------------------------------------------

        F.col("path")
        .alias("_source_path"),


        # ----------------------------------------------------
        # Filename
        # ----------------------------------------------------

        F.element_at(
            F.split(
                F.col("path"),
                "/"
            ),
            -1
        )
        .alias("_source_file"),


        # ----------------------------------------------------
        # File modification time
        # ----------------------------------------------------

        F.col("modificationTime")
        .alias("_file_modification_time")
    )
)


# ============================================================
# 10. START AUTO LOADER
# ============================================================
#
# availableNow=True means:
#
#     Process all currently available files
#     ↓
#     Stop the stream
#
# This is ideal for a daily scheduled Databricks Job.
#
# ============================================================

query = (

    stream_df

    .writeStream

    .foreachBatch(process_batch)

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    .option(
        "checkpointLocation",
        f"{CHECKPOINT_PATH}/stream"
    )

    # --------------------------------------------------------
    # Process available files and stop
    # --------------------------------------------------------

    .trigger(
        availableNow=True
    )

    .start()
)


# ============================================================
# 11. WAIT FOR STREAM TO FINISH
# ============================================================

query.awaitTermination()


# ============================================================
# 12. FINAL MESSAGE
# ============================================================

print()
print("=" * 70)
print("AUTO LOADER BRONZE INGESTION COMPLETED")
print("=" * 70)